In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sns.set(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
np.random.seed(101)
sys.path.append("..")
from src.evaluation import *
from sklearn.linear_model import LinearRegression

In [18]:
#load my data
data_path = Path("../data/processed/NVDA_feature_engineering.parquet")
df = pd.read_parquet(data_path)
df

,date,adj close,close,high,low,open,volume,ticker,daily_rtn,excess_return,outlier_iqr,outlier_winsorize,outlier_z,mom_5,log_return,vol_std_5,next_excess_return
0,2020-08-20,12.101113,12.141000,12.375000,11.878750,11.975000,921388000,NVDA,0.000206,-0.002953,False,-0.002953,False,NaN,NaN,NaN,0.041242
1,2020-08-21,12.641834,12.683500,12.808750,12.195250,12.201750,999868000,NVDA,0.044684,0.041242,False,0.041242,False,NaN,NaN,NaN,-0.007147
2,2020-08-24,12.678459,12.720250,12.912500,12.507500,12.883750,490564000,NVDA,0.002897,-0.007147,False,-0.007147,False,NaN,0.043714,NaN,-0.001257
3,2020-08-25,12.708115,12.750000,12.761250,12.573750,12.630750,289076000,NVDA,0.002339,-0.001257,False,-0.001257,False,NaN,0.002893,NaN,-0.008392
4,2020-08-26,12.731039,12.773000,12.868500,12.677750,12.799250,321244000,NVDA,0.001804,-0.008392,False,-0.008392,False,NaN,0.002336,NaN,-0.013006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,2025-08-12,183.160004,183.160004,184.479996,179.460007,182.960007,145485700,NVDA,0.006042,-0.005303,False,-0.005303,False,0.011444,-0.003509,0.016267,-0.011802
1211,2025-08-13,181.589996,181.589996,183.970001,179.350006,182.619995,179871700,NVDA,-0.008572,-0.011802,False,-0.011802,False,0.027488,0.006024,0.008541,0.002065
1212,2025-08-14,182.020004,182.020004,183.020004,179.460007,179.750000,129554000,NVDA,0.002368,0.002065,False,0.002065,False,0.012095,-0.008609,0.005305,-0.005728
1213,2025-08-15,180.449997,180.449997,181.899994,178.039993,181.880005,156602200,NVDA,-0.008625,-0.005728,False,-0.005728,False,0.006915,0.002365,0.008105,0.008746


In [19]:
#export directory
img_dir = Path('../deliverables/images')
img_dir.mkdir(parents=True, exist_ok=True)
def savefig(name):
    plt.tight_layout()
    plt.savefig(img_dir / name, dpi=300)
    print(f'Saved {name}')

In [20]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

target_col = "next_excess_return"
feature_col = "log_return"

results = []

methods = {
    "fill_median": df.copy(),
    "fill_mean": df.copy(),
    "dropna": df.copy()
}

methods["fill_median"][feature_col] = methods["fill_median"][feature_col].fillna(methods["fill_median"][feature_col].median())
methods["fill_median"][target_col] = methods["fill_median"][target_col].fillna(methods["fill_median"][target_col].median())
methods["fill_mean"][feature_col] = methods["fill_mean"][feature_col].fillna(methods["fill_mean"][feature_col].mean())
methods["fill_mean"][target_col] = methods["fill_mean"][target_col].fillna(methods["fill_mean"][target_col].mean())
methods["dropna"] = methods["dropna"].dropna(subset=[feature_col, target_col])

for method_name, df_m in methods.items():
    metrics = evaluate_model(df_m, feature_col, target_col)
    metrics["method"] = method_name
    results.append(metrics)

results_df = pd.DataFrame(results).set_index("method")
results_df

,slope,intercept,R2,MAE,n_samples
method,,,,,
fill_median,-0.017145,0.001611,0.000647,0.017406,1215
fill_mean,-0.017170,0.001611,0.000649,0.017406,1215
dropna,-0.017171,0.001586,0.000651,0.017408,1212
